# Honest Baseline: GroupKFold on Composite_Mix_ID

## Overview
This notebook mirrors the replication study structure but **removes the data leakage** introduced by the naive random split.

- **Data**: `combined_slurry_data_cleaned.csv` (both batches combined)
- **Split strategy**: `GroupKFold(n_splits=5)` grouped by `Composite_Mix_ID`
  - Every replicate of a given formulation lands entirely in **one** fold — never split across train/test
  - This is the correct evaluation protocol when replicates exist
- **Models**: XGBoost with paper-exact hyperparameters + cascading prediction architecture
- **Expected result**: R² scores significantly lower than the replication study, revealing the true generalisation ability of the models

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor

RANDOM_STATE = 42
N_SPLITS = 5

# Column definitions (identical to replication study)
TARGET_1   = "Viscosity_at_shear_rate_1_1/s"
TARGET_10  = "Viscosity_at_shear_rate_10_1/s"
TARGET_100 = "Viscosity_at_shear_rate_100_1/s"
TARGET_COLS = [TARGET_1, TARGET_10, TARGET_100]

CAT_COLS  = ["Formulation", "Dispersent_Type"]
CONT_COLS = ["Solid_Content_pct", "Solid_Additive_pct"]
GROUP_COL = "Composite_Mix_ID"
BATCH_COL = "Source_Batch"

DATA_PATH = Path("../data/processed/combined_slurry_data_cleaned.csv")
df = pd.read_csv(DATA_PATH)

print(f"Total rows: {len(df)}")
print(f"Unique Composite_Mix_IDs: {df[GROUP_COL].nunique()}")
print(f"Batch composition: {df[BATCH_COL].value_counts().to_dict()}")
print(df.head())

Total rows: 178
Unique Composite_Mix_IDs: 68
Batch composition: {'Batch_1': 91, 'Batch_2': 87}
  Formulation Dispersent_Type  Solid_Content_pct  Solid_Additive_pct  \
0          F1        Hypermer               73.0                0.50   
1          F1        Hypermer               73.0                0.25   
2          F1        Hypermer               73.0                0.25   
3          F1        Hypermer               77.0                0.25   
4          F1        Hypermer               73.0                0.25   

   Viscosity_at_shear_rate_1_1/s  Viscosity_at_shear_rate_10_1/s  \
0                       10.56640                         3.78921   
1                       71.65190                        14.08460   
2                        9.64639                         3.26827   
3                       61.11070                        18.77450   
4                        8.37111                         4.83186   

   Viscosity_at_shear_rate_100_1/s Source_Batch       Composite

## GroupKFold Split

`GroupKFold` ensures that **no `Composite_Mix_ID` appears in both train and test** within any fold.
Because all replicates of a formulation share the same ID, their identical (or near-identical) rows
will never leak into the opposite set — this is the critical difference from the paper's naive split.

In [2]:
# One-hot encode categoricals once on the full dataset, then let each fold
# fit its own scaler strictly on the training rows.
X_raw = pd.get_dummies(df[CAT_COLS + CONT_COLS], columns=CAT_COLS, drop_first=False)
y     = df[TARGET_COLS]
groups = df[GROUP_COL]

gkf = GroupKFold(n_splits=N_SPLITS)

# Verify no group leaks across folds
print(f"GroupKFold with {N_SPLITS} splits on {groups.nunique()} unique groups")
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_raw, y, groups), start=1):
    tr_groups = set(groups.iloc[tr_idx])
    te_groups = set(groups.iloc[te_idx])
    overlap   = tr_groups & te_groups
    print(f"  Fold {fold}: train={len(tr_idx)} rows ({len(tr_groups)} groups) | "
          f"test={len(te_idx)} rows ({len(te_groups)} groups) | overlap={len(overlap)}")

GroupKFold with 5 splits on 68 unique groups
  Fold 1: train=142 rows (54 groups) | test=36 rows (14 groups) | overlap=0
  Fold 2: train=142 rows (54 groups) | test=36 rows (14 groups) | overlap=0
  Fold 3: train=142 rows (54 groups) | test=36 rows (14 groups) | overlap=0
  Fold 4: train=143 rows (55 groups) | test=35 rows (13 groups) | overlap=0
  Fold 5: train=143 rows (55 groups) | test=35 rows (13 groups) | overlap=0


## XGBoost + Cascading Prediction (Paper Hyperparameters)

Architecture is identical to the replication study:
1. Train on 4 features → predict **10 1/s**
2. Append predicted 10 1/s as a 5th feature
3. Train two fresh models on 5 features → predict **1 1/s** and **100 1/s**

The only change is the train/test split: each fold is now group-safe.

In [3]:
xgb_params = dict(
    learning_rate=0.1,
    max_depth=3,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    n_estimators=100,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
)

fold_results = []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_raw, y, groups), start=1):
    X_tr_raw, X_te_raw = X_raw.iloc[tr_idx].copy(), X_raw.iloc[te_idx].copy()
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    # Fit scaler on training fold only
    scaler = StandardScaler()
    X_tr_raw[CONT_COLS] = scaler.fit_transform(X_tr_raw[CONT_COLS])
    X_te_raw[CONT_COLS] = scaler.transform(X_te_raw[CONT_COLS])

    # --- Step 1: predict 10 1/s ---
    m10 = XGBRegressor(**xgb_params)
    m10.fit(X_tr_raw, y_tr[TARGET_10])

    # --- Step 2: inject predicted 10 1/s as 5th feature ---
    X_tr_cas = X_tr_raw.copy()
    X_te_cas = X_te_raw.copy()
    X_tr_cas["Predicted_Visc_10"] = m10.predict(X_tr_raw)
    X_te_cas["Predicted_Visc_10"] = m10.predict(X_te_raw)

    # --- Step 3: predict 1 1/s and 100 1/s from 5 features ---
    m1   = XGBRegressor(**xgb_params)
    m100 = XGBRegressor(**xgb_params)
    m1.fit(X_tr_cas,   y_tr[TARGET_1])
    m100.fit(X_tr_cas, y_tr[TARGET_100])

    preds = {
        TARGET_10:  m10.predict(X_te_raw),
        TARGET_1:   m1.predict(X_te_cas),
        TARGET_100: m100.predict(X_te_cas),
    }

    for target, pred in preds.items():
        true = y_te[target].values
        fold_results.append({
            "fold":  fold,
            "target": target,
            "R2":   r2_score(true, pred),
            "RMSE": np.sqrt(mean_squared_error(true, pred)),
            "MAE":  mean_absolute_error(true, pred),
        })

results_df = pd.DataFrame(fold_results)
print(results_df.to_string(index=False))

 fold                          target        R2      RMSE       MAE
    1  Viscosity_at_shear_rate_10_1/s  0.863755  3.995382  3.072434
    1   Viscosity_at_shear_rate_1_1/s  0.717150 29.723229 20.492356
    1 Viscosity_at_shear_rate_100_1/s  0.914892  0.967213  0.768763
    2  Viscosity_at_shear_rate_10_1/s  0.934297  3.284890  2.294661
    2   Viscosity_at_shear_rate_1_1/s  0.820016 30.742292 13.262609
    2 Viscosity_at_shear_rate_100_1/s  0.887336  1.163244  0.843126
    3  Viscosity_at_shear_rate_10_1/s  0.806927  5.877119  3.842074
    3   Viscosity_at_shear_rate_1_1/s  0.748548 40.262945 24.761619
    3 Viscosity_at_shear_rate_100_1/s  0.821167  1.441905  1.011346
    4  Viscosity_at_shear_rate_10_1/s  0.367610  5.846156  4.332139
    4   Viscosity_at_shear_rate_1_1/s -0.505659 59.281862 40.999068
    4 Viscosity_at_shear_rate_100_1/s  0.752678  1.291402  0.930642
    5  Viscosity_at_shear_rate_10_1/s  0.814975  5.396429  3.721220
    5   Viscosity_at_shear_rate_1_1/s  0.471795 

## Summary: Mean ± Std across Folds

In [4]:
summary = (
    results_df
    .groupby("target")[["R2", "RMSE", "MAE"]]
    .agg(["mean", "std"])
    .round(4)
)

# Flatten multi-index columns
summary.columns = ["_".join(c) for c in summary.columns]
summary = summary.reset_index()

print("XGBoost + Cascading  |  GroupKFold (n=5, grouped by Composite_Mix_ID)")
print("=" * 70)
for _, row in summary.iterrows():
    print(f"\n{row['target']}")
    print(f"  R2:   {row['R2_mean']:.4f}  ±  {row['R2_std']:.4f}")
    print(f"  RMSE: {row['RMSE_mean']:.4f}  ±  {row['RMSE_std']:.4f}")
    print(f"  MAE:  {row['MAE_mean']:.4f}  ±  {row['MAE_std']:.4f}")

print("\n")
print("COMPARE with replication_study.ipynb (naive 80/20 split):")
naive = {TARGET_10: 0.8834, TARGET_1: None, TARGET_100: None}
print("  10 1/s naive R2: ~0.88  vs  GroupKFold R2:",
      f"{summary.loc[summary.target==TARGET_10, 'R2_mean'].values[0]:.4f}")

XGBoost + Cascading  |  GroupKFold (n=5, grouped by Composite_Mix_ID)

Viscosity_at_shear_rate_100_1/s
  R2:   0.8480  ±  0.0634
  RMSE: 1.2219  ±  0.1747
  MAE:  0.8949  ±  0.0923

Viscosity_at_shear_rate_10_1/s
  R2:   0.7575  ±  0.2238
  RMSE: 4.8800  ±  1.1749
  MAE:  3.4525  ±  0.7878

Viscosity_at_shear_rate_1_1/s
  R2:   0.4504  ±  0.5503
  RMSE: 41.2639  ±  12.1963
  MAE:  26.2091  ±  10.6000


COMPARE with replication_study.ipynb (naive 80/20 split):
  10 1/s naive R2: ~0.88  vs  GroupKFold R2: 0.7575
